# Login Behavior Analysis
### Access Log Anomaly Detection with Machine Learning and Matplotlib

This notebook analyzes the uploaded `Access-Log-Anomaly-Detection-Dataset.csv` dataset. It includes:

1. Dataset inspection and data quality checks
2. Exploratory data analysis with Matplotlib
3. Classification with Logistic Regression, Decision Tree, and Random Forest
4. Classification metrics and confusion-matrix visualizations
5. K-Means clustering and cluster visualizations
6. A Random Forest prediction example for a new login behavior

**Dataset details observed:** 5,000 rows and 36 columns, including the `target` label. The notebook will also work with a compatible CSV containing the same target and relevant feature columns.

> Run the cells from top to bottom. The notebook is designed for Google Colab, but can also run locally after adjusting the dataset path.

## Step 1 — Import libraries and configure the notebook

We use pandas and NumPy for data handling, Matplotlib for charts, and scikit-learn for preprocessing and machine learning. No Seaborn dependency is required.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    silhouette_score
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 1200)

print("Libraries imported successfully.")

## Step 2 — Load the dataset

In Google Colab, the cell first checks `/content/Access-Log-Anomaly-Detection-Dataset.csv` and `/content/data/Access-Log-Anomaly-Detection-Dataset.csv`. If neither exists, it opens a file-upload prompt.

If running locally, update `DATASET_PATH` to the CSV's location.

In [ ]:
DATASET_FILENAME = "Access-Log-Anomaly-Detection-Dataset.csv"
possible_paths = [
    f"/content/{DATASET_FILENAME}",
    f"/content/data/{DATASET_FILENAME}",
    DATASET_FILENAME,
    f"/mnt/data/{DATASET_FILENAME}"
]

DATASET_PATH = next(
    (path for path in possible_paths if os.path.exists(path)),
    None
)

if DATASET_PATH is None:
    try:
        from google.colab import files
        print("Please upload:", DATASET_FILENAME)
        uploaded = files.upload()
        uploaded_name = next(iter(uploaded))
        DATASET_PATH = uploaded_name
    except ImportError:
        raise FileNotFoundError(
            f"Could not find {DATASET_FILENAME}. "
            "Set DATASET_PATH to the CSV file location."
        )

df = pd.read_csv(DATASET_PATH)

print("Dataset loaded successfully.")
print("Path:", DATASET_PATH)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())

## Step 3 — Validate and inspect the dataset

This step checks the required `target` column, data types, missing values, duplicates, and class distribution. The target labels are normalized to lowercase and whitespace is removed.

In [ ]:
if "target" not in df.columns:
    raise ValueError("The dataset must contain a 'target' column.")

df["target"] = df["target"].astype(str).str.lower().str.strip()

print("=" * 70)
print("DATASET SUMMARY")
print("=" * 70)
print(f"Rows               : {df.shape[0]:,}")
print(f"Columns            : {df.shape[1]}")
print(f"Duplicate records  : {df.duplicated().sum():,}")
print(f"Total missing cells: {df.isnull().sum().sum():,}")

dtype_summary = pd.DataFrame({
    "Column Name": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isnull().sum().values,
    "Unique Values": [df[col].nunique(dropna=True) for col in df.columns]
})
display(dtype_summary)

print("\nTarget distribution:")
target_summary = df["target"].value_counts(dropna=False).to_frame("Count")
target_summary["Percentage"] = (
    df["target"].value_counts(normalize=True, dropna=False) * 100
).round(2)
display(target_summary)

## Step 4 — Exploratory data analysis with Matplotlib

The following charts show:
- The number of benign and malicious records
- Missing values by column (only columns with missing data)
- Correlations between numerical features

Correlation indicates association, not causation.

In [ ]:
# 4.1 Target class distribution
target_counts = df["target"].value_counts()

plt.figure(figsize=(8, 5))
bars = plt.bar(target_counts.index.astype(str), target_counts.values)
plt.title("Benign vs Malicious Login Distribution")
plt.xlabel("Target Class")
plt.ylabel("Number of Records")
plt.grid(axis="y", linestyle="--", alpha=0.4)

for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{int(bar.get_height()):,}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

# 4.2 Missing values
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]

if not missing.empty:
    plt.figure(figsize=(11, 5))
    plt.bar(missing.index.astype(str), missing.values)
    plt.title("Missing Values by Feature")
    plt.xlabel("Feature")
    plt.ylabel("Missing Value Count")
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found.")

# 4.3 Numerical correlation heatmap
numeric_df = df.select_dtypes(include=np.number)

if numeric_df.shape[1] >= 2:
    corr = numeric_df.corr()

    fig, ax = plt.subplots(figsize=(13, 9))
    image = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=90)
    ax.set_yticklabels(corr.columns)
    ax.set_title("Numerical Feature Correlation Heatmap")
    fig.colorbar(image, ax=ax, label="Correlation coefficient")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numerical columns for a correlation heatmap.")

## Step 5 — Prepare features and split the data

The `target` column is the label to predict. We normalize the labels and retain the `benign` and `malicious` classes. Numerical values are imputed and scaled; categorical values are imputed and one-hot encoded.

The train/test split is 80%/20%, with stratification to preserve the class proportions. Preprocessing is fitted on the training data only to reduce data leakage.

In [ ]:
valid_mask = df["target"].isin(["benign", "malicious"])
model_df = df.loc[valid_mask].copy()

if model_df["target"].nunique() != 2:
    raise ValueError(
        "The target must contain both 'benign' and 'malicious' classes."
    )
if model_df["target"].value_counts().min() < 2:
    raise ValueError("Each target class must have at least two records.")

X = model_df.drop(columns=["target"]).copy()
y = model_df["target"].copy()

# Treat boolean, object, and category columns as categorical.
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:  # Compatibility with older scikit-learn versions
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

transformers = []
if num_cols:
    transformers.append((
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        num_cols
    ))

if cat_cols:
    transformers.append((
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", encoder)
        ]),
        cat_cols
    ))

if not transformers:
    raise ValueError("No usable feature columns were found.")

preprocessor = ColumnTransformer(transformers=transformers)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training records:", len(X_train))
print("Testing records :", len(X_test))
print("Numerical features:", len(num_cols))
print("Categorical features:", len(cat_cols))
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape :", X_test_processed.shape)

## Step 6 — Train classification models

Three classifiers are trained on the same training split:
- Logistic Regression
- Decision Tree
- Random Forest

We evaluate accuracy, precision, recall, and F1-score. For precision, recall, and F1-score, `malicious` is treated as the positive class.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=10,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    )
}

trained_models = {}
performance_summary = {}
confusion_matrices = {}
predictions = {}

for name, model in models.items():
    print("\n" + "=" * 65)
    print(name.upper().center(65))
    print("=" * 65)

    model.fit(X_train_processed, y_train)
    y_pred = model.predict(X_test_processed)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, pos_label="malicious", zero_division=0)
    rec = recall_score(y_test, y_pred, pos_label="malicious", zero_division=0)
    f1 = f1_score(y_test, y_pred, pos_label="malicious", zero_division=0)

    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=["benign", "malicious"]
    )

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}")

    print("\nConfusion Matrix (rows=actual, columns=predicted)")
    print("Label order: benign, malicious")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        labels=["benign", "malicious"],
        target_names=["benign", "malicious"],
        digits=4,
        zero_division=0
    ))

    trained_models[name] = model
    predictions[name] = y_pred
    confusion_matrices[name] = cm
    performance_summary[name] = {
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1
    }

performance_df = pd.DataFrame(performance_summary).T
display(performance_df.round(4))

## Step 7 — Visualize classification results

The first plot compares model metrics. The following confusion matrices show correct and incorrect predictions for each class:
- Top-left: benign correctly predicted as benign
- Top-right: benign incorrectly predicted as malicious
- Bottom-left: malicious incorrectly predicted as benign
- Bottom-right: malicious correctly predicted as malicious

In [ ]:
# 7.1 Model metric comparison
ax = performance_df.plot(kind="bar", figsize=(12, 6), ylim=(0, 1.05))
ax.set_title("Classification Model Performance Comparison")
ax.set_xlabel("Model")
ax.set_ylabel("Score")
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.xticks(rotation=15, ha="right")
plt.legend(title="Metric")
plt.tight_layout()
plt.show()

# 7.2 Confusion matrices
model_names = list(confusion_matrices.keys())
fig, axes = plt.subplots(1, len(model_names), figsize=(6 * len(model_names), 5), squeeze=False)
axes = axes.ravel()

for ax, name in zip(axes, model_names):
    cm = confusion_matrices[name]
    image = ax.imshow(cm, cmap="Blues")
    ax.set_title(name)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("Actual class")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Benign", "Malicious"])
    ax.set_yticklabels(["Benign", "Malicious"])

    threshold = cm.max() / 2 if cm.size else 0
    for i in range(2):
        for j in range(2):
            ax.text(
                j, i, f"{cm[i, j]:,}",
                ha="center", va="center",
                color="white" if cm[i, j] > threshold else "black"
            )
    fig.colorbar(image, ax=ax)

fig.suptitle("Confusion Matrix Comparison")
plt.tight_layout()
plt.show()

## Step 8 — K-Means clustering

Clustering is unsupervised, so the `target` label is excluded. The notebook uses available numeric behavioral features from the candidate list, imputes missing values, removes constant columns, and scales the data.

K-Means is tested for `K = 2` through `K = 6` (where possible). The silhouette score is used as one indicator of cluster separation. It does **not** mean that any cluster is automatically malicious.

In [ ]:
candidate_features = [
    "login_attempt_count",
    "anomaly_score",
    "log_hour",
    "time_diff_prev",
    "user_activity_frequency",
    "unique_actions",
    "location_count",
    "session_duration",
    "event_count_session",
    "session_complexity"
]

cluster_features = [
    col for col in candidate_features
    if col in model_df.columns
    and pd.api.types.is_numeric_dtype(model_df[col])
]

if len(cluster_features) < 2:
    raise ValueError(
        "At least two numeric candidate clustering features are required."
    )

X_cluster = model_df[cluster_features].copy()
X_cluster = X_cluster.replace([np.inf, -np.inf], np.nan)
X_cluster = X_cluster.dropna(axis=1, how="all")
X_cluster = X_cluster.fillna(X_cluster.median())

cluster_features = [
    col for col in X_cluster.columns
    if X_cluster[col].nunique() > 1
]
X_cluster = X_cluster[cluster_features]

if len(cluster_features) < 2:
    raise ValueError("At least two non-constant clustering features are required.")

if len(X_cluster) < 3:
    raise ValueError("At least three records are required for clustering.")

cluster_scaler = StandardScaler()
X_cluster_scaled = cluster_scaler.fit_transform(X_cluster)

silhouette_scores = {}
max_k = min(6, len(X_cluster) - 1)

for k in range(2, max_k + 1):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_cluster_scaled)

    if len(np.unique(labels)) < 2 or len(np.unique(labels)) >= len(X_cluster):
        continue

    score = silhouette_score(
        X_cluster_scaled,
        labels,
        sample_size=min(3000, len(X_cluster)),
        random_state=42
    )
    silhouette_scores[k] = score
    print(f"K = {k} | Silhouette Score = {score:.4f}")

if not silhouette_scores:
    raise ValueError("Could not calculate a valid silhouette score.")

best_k = max(silhouette_scores, key=silhouette_scores.get)
print(f"\nSelected K: {best_k}")
print(f"Best silhouette score: {silhouette_scores[best_k]:.4f}")

final_kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = final_kmeans.fit_predict(X_cluster_scaled)

df_cluster = model_df.copy()
df_cluster["Cluster"] = cluster_labels

cluster_counts = df_cluster["Cluster"].value_counts().sort_index()
cluster_summary = pd.DataFrame({
    "Record Count": cluster_counts,
    "Percentage": (cluster_counts / len(df_cluster) * 100).round(2)
})
display(cluster_summary)

print("Average behavioral feature values per cluster:")
display(df_cluster.groupby("Cluster")[cluster_features].mean().round(3))

## Step 9 — Visualize clustering results

These plots show the silhouette score for each tested `K`, the number of records assigned to each cluster, and the mean values of the selected behavioral features per cluster.

In [ ]:
# 9.1 Silhouette scores
plt.figure(figsize=(8, 5))
k_values = list(silhouette_scores.keys())
scores = list(silhouette_scores.values())
plt.plot(k_values, scores, marker="o", linewidth=2)
plt.title("K-Means: Silhouette Score vs K")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Silhouette score")
plt.xticks(k_values)
plt.grid(True, linestyle="--", alpha=0.4)

for k, score in silhouette_scores.items():
    plt.annotate(
        f"{score:.3f}",
        (k, score),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center"
    )

plt.tight_layout()
plt.show()

# 9.2 Cluster sizes
plt.figure(figsize=(8, 5))
bars = plt.bar(cluster_counts.index.astype(str), cluster_counts.values)
plt.title("Number of Records in Each Cluster")
plt.xlabel("Cluster")
plt.ylabel("Record count")
plt.grid(axis="y", linestyle="--", alpha=0.4)

for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{int(bar.get_height()):,}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

# 9.3 Mean feature values by cluster
cluster_means = df_cluster.groupby("Cluster")[cluster_features].mean()
ax = cluster_means.plot(kind="bar", figsize=(13, 6))
ax.set_title("Average Behavioral Feature Values by Cluster")
ax.set_xlabel("Cluster")
ax.set_ylabel("Average feature value")
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.xticks(rotation=0)
plt.legend(title="Feature", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Step 10 — Predict a new login behavior

The prediction pipeline uses the seven behavioral fields below. The example values are defaults only; replace them with values appropriate to the record you want to test.

The model is trained on the available dataset for this demonstration. For a formal evaluation, assess the prediction pipeline on a held-out test set as well.

In [ ]:
prediction_features = [
    "login_attempt_count",
    "anomaly_score",
    "session_duration",
    "event_count_session",
    "session_complexity",
    "user_activity_frequency",
    "unique_actions"
]

missing_prediction_features = [
    col for col in prediction_features
    if col not in model_df.columns
]

if missing_prediction_features:
    raise ValueError(
        "Missing prediction features: "
        + ", ".join(missing_prediction_features)
    )

X_behavior = model_df[prediction_features].copy()

for col in prediction_features:
    X_behavior[col] = pd.to_numeric(X_behavior[col], errors="coerce")

X_behavior = X_behavior.replace([np.inf, -np.inf], np.nan)

prediction_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("rf", RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ))
])

prediction_pipeline.fit(X_behavior, model_df["target"])

print("Prediction model trained.")

## Step 11 — Enter a sample login record

Edit the values in the dictionary and run this cell to classify one record. The chart shows the model's estimated probability for each class.

If you want to try several records, change the dictionary and run the cell again.

In [ ]:
new_login = {
    "login_attempt_count": 1,
    "anomaly_score": 0.15,
    "session_duration": 650,
    "event_count_session": 40,
    "session_complexity": 3.8,
    "user_activity_frequency": 12,
    "unique_actions": 8
}

new_login_df = pd.DataFrame([new_login], columns=prediction_features)

predicted_class = prediction_pipeline.predict(new_login_df)[0]
predicted_probabilities = prediction_pipeline.predict_proba(new_login_df)[0]
class_names = list(prediction_pipeline.classes_)

benign_probability = (
    predicted_probabilities[class_names.index("benign")] * 100
)
malicious_probability = (
    predicted_probabilities[class_names.index("malicious")] * 100
)

print("=" * 60)
print("LOGIN BEHAVIOR PREDICTION".center(60))
print("=" * 60)
print("Predicted class:", predicted_class.upper())
print(f"Benign probability    : {benign_probability:.2f}%")
print(f"Malicious probability : {malicious_probability:.2f}%")

plt.figure(figsize=(8, 5))
bars = plt.bar(
    ["Benign", "Malicious"],
    [benign_probability, malicious_probability]
)
plt.title("Login Behavior Prediction Probability")
plt.xlabel("Predicted class")
plt.ylabel("Probability (%)")
plt.ylim(0, 110)
plt.grid(axis="y", linestyle="--", alpha=0.4)

for bar, value in zip(
    bars,
    [benign_probability, malicious_probability]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{value:.2f}%",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

## Step 12 — Final summary

This notebook has:
- Inspected the access-log dataset and target distribution
- Visualized missing values and numerical correlations
- Trained and compared three classification algorithms
- Displayed classification metrics and confusion matrices
- Performed K-Means clustering and visualized cluster summaries
- Predicted a new login behavior and charted its class probabilities

### Interpretation reminder

A high malicious probability is a model estimate, not proof of an attack. Review model performance, false positives, false negatives, data quality, and the dataset's label-generation process before using the model in an operational security system.